After this cell finishes, restart the Colab runtime before running imports:

Runtime → Restart session

Colab may print dependency warnings for preinstalled packages like gradio, jax, opencv, shap, or rasterio. Those are expected and can be ignored as long as the verification cell below imports successfully.


In [ ]:
%pip install -q --upgrade --force-reinstall --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  scipy==1.13.1 \
  scikit-learn==1.5.1 \
  pyarrow==17.0.0 \
  fsspec==2024.6.1 \
  pillow==11.3.0 \
  requests==2.32.4 \
  jedi==0.19.1 \
  transformers==4.44.2 \
  accelerate==0.33.0 \
  bitsandbytes==0.43.3 \
  datasets==2.21.0 \
  evaluate==0.4.2 \
  matplotlib==3.9.2 \
  seaborn==0.13.2 \
  kaggle==1.6.17 \
  huggingface_hub==0.25.2 \
  wandb==0.17.5


In [ ]:
import transformers
import datasets
import evaluate
import sklearn
import pandas
import numpy
import scipy
import pyarrow
import PIL
import matplotlib
import huggingface_hub
import fsspec

print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("evaluate", evaluate.__version__)
print("sklearn", sklearn.__version__)
print("pandas", pandas.__version__)
print("numpy", numpy.__version__)
print("scipy", scipy.__version__)
print("pyarrow", pyarrow.__version__)
print("pillow", PIL.__version__)
print("matplotlib", matplotlib.__version__)
print("huggingface_hub", huggingface_hub.__version__)
print("fsspec", fsspec.__version__)
print("Environment ready.")


In [ ]:
from pathlib import Path
import shutil

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
target = kaggle_dir / "kaggle.json"

if not target.exists():
    print("Upload your Kaggle API token file named kaggle.json.")
    if files is None:
        raise RuntimeError("google.colab.files is unavailable. Place kaggle.json at ~/.kaggle/kaggle.json manually.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Expected an uploaded file named kaggle.json")
    shutil.move("kaggle.json", target)

if target.exists():
    target.chmod(0o600)

print("Kaggle token exists:", target.exists())
print("Kaggle token path:", target)


In [ ]:
from pathlib import Path

kaggle_path = Path.home() / ".kaggle" / "kaggle.json"
print("Kaggle token exists:", kaggle_path.exists())


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
# Persistent Colab handoff paths for H9-H16.
# Google Drive is the source of truth across separate Colab notebooks/runtimes.
from pathlib import Path

USE_GOOGLE_DRIVE = True

try:
    from google.colab import drive
    if USE_GOOGLE_DRIVE:
        drive.mount("/content/drive")
        LOCAL_UNBLOCKER_ROOT = Path("/content/drive/MyDrive/GemScan/local_unblocker")
    else:
        LOCAL_UNBLOCKER_ROOT = Path("/content/GemScan/notebooks/data/local_unblocker")
except ModuleNotFoundError:
    # Local fallback for VS Code/Jupyter outside Colab.
    LOCAL_UNBLOCKER_ROOT = Path("notebooks/data/local_unblocker")

RAW_DIR = LOCAL_UNBLOCKER_ROOT / "raw"
PROCESSED_DIR = LOCAL_UNBLOCKER_ROOT / "processed"
SCRUBBED_DIR = LOCAL_UNBLOCKER_ROOT / "scrubbed"
RESULTS_DIR = LOCAL_UNBLOCKER_ROOT / "results"
FIXTURES_DIR = LOCAL_UNBLOCKER_ROOT / "fixtures"

for path in [RAW_DIR, PROCESSED_DIR, SCRUBBED_DIR, RESULTS_DIR, FIXTURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("LOCAL_UNBLOCKER_ROOT", LOCAL_UNBLOCKER_ROOT)
print("RAW_DIR", RAW_DIR)
print("PROCESSED_DIR", PROCESSED_DIR)
print("SCRUBBED_DIR", SCRUBBED_DIR)
print("RESULTS_DIR", RESULTS_DIR)
print("FIXTURES_DIR", FIXTURES_DIR)


# H9.1 PULL UCI SMS SPAM COLLECTION


In [ ]:
from pathlib import Path
import zipfile
import urllib.request
import pandas as pd

# Uses RAW_DIR and PROCESSED_DIR from the persistent Drive path setup cell.
uci_raw_dir = RAW_DIR / "uci_sms_spam"
uci_raw_dir.mkdir(parents=True, exist_ok=True)

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
zip_path = uci_raw_dir / "smsspamcollection.zip"

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(uci_raw_dir)

sms_path = uci_raw_dir / "SMSSpamCollection"

df = pd.read_csv(
    sms_path,
    sep="	",
    header=None,
    names=["label", "text"],
    encoding="latin-1",
)

df["id"] = [f"uci-{i:05d}" for i in range(len(df))]
df["source"] = "uci_sms_spam_collection"
df["language"] = "en"
df["split"] = "unassigned"

df = df[["id", "source", "text", "label", "language", "split"]]

out_path = PROCESSED_DIR / "uci_sms_clean.csv"
df.to_csv(out_path, index=False)

print(df.shape)
print(df["label"].value_counts())
print("saved", out_path)
df.head()


# H9.2 Kaggle SMS-Spam datasets

In [ ]:
from pathlib import Path
import zipfile
import subprocess

# Uses RAW_DIR from the persistent Drive path setup cell.
kaggle_raw_dir = RAW_DIR / "kaggle"
kaggle_raw_dir.mkdir(parents=True, exist_ok=True)

datasets = [
    "abhishek14398/sms-spam-collection",
    "vishakhdapat/sms-spam-detection-dataset",
]

for dataset in datasets:
    owner, name = dataset.split("/")
    out_dir = kaggle_raw_dir / name
    out_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", dataset,
            "-p", str(out_dir),
            "--force",
        ],
        check=True,
    )

    for zip_path in out_dir.glob("*.zip"):
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(out_dir)

    print("Downloaded:", dataset)
    print("Files:", [p.name for p in out_dir.iterdir()])


In [ ]:
import pandas as pd

for csv_path in (RAW_DIR / "kaggle").glob("**/*.csv"):
    print()
    print(csv_path)
    try:
        preview = pd.read_csv(csv_path, encoding="utf-8")
    except UnicodeDecodeError:
        preview = pd.read_csv(csv_path, encoding="latin-1")

    print(preview.shape)
    print(preview.columns.tolist())
    display(preview.head())


# H9.3 Multilingual scam corpus

In [ ]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

# Uses RAW_DIR and PROCESSED_DIR from the persistent Drive path setup cell.
multilingual_raw_dir = RAW_DIR / "multilingual"
multilingual_raw_dir.mkdir(parents=True, exist_ok=True)

dataset_id = "dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset"

ds = load_dataset(dataset_id, split="train")
df = ds.to_pandas()

raw_snapshot_path = multilingual_raw_dir / "sms_spam_multilingual_snapshot.csv"
df.to_csv(raw_snapshot_path, index=False)

print(df.shape)
print(df.columns.tolist())
print("saved raw snapshot", raw_snapshot_path)
display(df.head())


# H9.5 - PII Scrub all corpora

In [ ]:
import json
import re
import shutil
from pathlib import Path

import pandas as pd

# Uses LOCAL_UNBLOCKER_ROOT / RAW_DIR / PROCESSED_DIR / SCRUBBED_DIR / RESULTS_DIR / FIXTURES_DIR
# from the persistent Drive path setup cell. CSVs written here survive separate Colab notebooks.
for directory in [RAW_DIR, PROCESSED_DIR, SCRUBBED_DIR, RESULTS_DIR, FIXTURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Copy versioned synthetic fixtures from the repo into Drive so H11/H14/H15 can read them in separate runtimes.
repo_fixture_candidates = [
    Path("notebooks/local_unblocker/fixtures/hard_cases_v0.csv"),
    Path("GemScan/notebooks/local_unblocker/fixtures/hard_cases_v0.csv"),
]
for candidate in repo_fixture_candidates:
    if candidate.exists():
        shutil.copy2(candidate, FIXTURES_DIR / candidate.name)
        print("copied fixture to Drive", FIXTURES_DIR / candidate.name)
        break

PII_PATTERNS = {
    "email": re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.IGNORECASE),
    "url": re.compile(r"\b(?:https?://|www\.)\S+", re.IGNORECASE),
    "credit_or_long_payment_number": re.compile(r"(?<!\w)(?:\d[ -]?){13,19}(?!\w)"),
    "phone_number": re.compile(r"(?<!\w)(?:\+?\d[\d\s().-]{7,}\d)(?!\w)"),
    "crypto_wallet": re.compile(r"\b(?:0x[a-fA-F0-9]{40}|bc1[a-zA-HJ-NP-Z0-9]{25,59}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b"),
}

REPLACEMENTS = {
    "email": "[EMAIL_REDACTED]",
    "url": "[URL_REDACTED]",
    "credit_or_long_payment_number": "[PAYMENT_NUMBER_REDACTED]",
    "phone_number": "[PHONE_REDACTED]",
    "crypto_wallet": "[WALLET_REDACTED]",
}

TEXT_COLUMN_CANDIDATES = ["text", "message", "sms", "v2", "Text", "Message", "SMS", "body", "Body"]
LABEL_COLUMN_CANDIDATES = ["label", "labels", "category", "v1", "Label", "Category", "class", "Class"]


def read_csv_flexible(path: Path) -> pd.DataFrame:
    for encoding in ["utf-8", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="latin-1", errors="replace")


def redact_text(value, counts):
    if pd.isna(value):
        return value
    text = str(value)
    for key, pattern in PII_PATTERNS.items():
        text, n = pattern.subn(REPLACEMENTS[key], text)
        counts[key] += n
    text = re.sub(r"\s+", " ", text).strip()
    return text


def scrub_dataframe(frame: pd.DataFrame):
    counts = {key: 0 for key in PII_PATTERNS}
    scrubbed = frame.copy()
    object_columns = scrubbed.select_dtypes(include=["object", "string"]).columns
    for column in object_columns:
        scrubbed[column] = scrubbed[column].map(lambda value: redact_text(value, counts))
    return scrubbed, counts


def first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def normalize_for_combined(frame: pd.DataFrame, source_name: str) -> pd.DataFrame | None:
    text_col = first_existing(frame.columns, TEXT_COLUMN_CANDIDATES)
    label_col = first_existing(frame.columns, LABEL_COLUMN_CANDIDATES)
    if text_col is None or label_col is None:
        return None
    normalized = pd.DataFrame(
        {
            "id": [f"{source_name}-{i:06d}" for i in range(len(frame))],
            "source": source_name,
            "text": frame[text_col].astype(str),
            "label": frame[label_col].astype(str).str.lower().str.strip(),
            "language": frame["language"].astype(str) if "language" in frame.columns else "en",
            "split": frame["split"].astype(str) if "split" in frame.columns else "unassigned",
        }
    )
    normalized["label"] = normalized["label"].replace({"0": "ham", "1": "spam", "legitimate": "ham"})
    return normalized[normalized["text"].str.strip().ne("")]


def safe_output_name(path: Path) -> str:
    try:
        relative = path.relative_to(LOCAL_UNBLOCKER_ROOT)
    except ValueError:
        relative = path
    parts = [part for part in relative.with_suffix("").parts if part not in {"raw", "processed", "fixtures", "scrubbed"}]
    return "__".join(parts).replace(" ", "_")


summary_rows = []
combined_frames = []
input_paths = []

for base_dir in [RAW_DIR, PROCESSED_DIR, FIXTURES_DIR]:
    if base_dir.exists():
        input_paths.extend(path for path in base_dir.glob("**/*.csv") if "scrubbed" not in path.name.lower())

for csv_path in sorted(set(input_paths)):
    frame = read_csv_flexible(csv_path)
    scrubbed, counts = scrub_dataframe(frame)
    output_name = f"{safe_output_name(csv_path)}__scrubbed.csv"
    output_path = SCRUBBED_DIR / output_name
    scrubbed.to_csv(output_path, index=False)

    source_name = safe_output_name(csv_path)
    normalized = normalize_for_combined(scrubbed, source_name)
    if normalized is not None:
        combined_frames.append(normalized)

    summary_rows.append(
        {
            "input_path": str(csv_path),
            "output_path": str(output_path),
            "rows": len(frame),
            **counts,
        }
    )

# If the H9.3 multilingual Hugging Face dataframe is still in memory, create the selected-language seed now.
hf_multilingual = globals().get("df")
if isinstance(hf_multilingual, pd.DataFrame):
    language_columns = {
        "en": "text",
        "es": "text_es",
        "hi": "text_hi",
        "zh-Hans": "text_zh",
        "ja": "text_ja",
    }
    if "labels" in hf_multilingual.columns and all(column in hf_multilingual.columns for column in language_columns.values()):
        multilingual_rows = []
        for row_index, row in hf_multilingual.iterrows():
            for language, column in language_columns.items():
                value = row[column]
                if pd.isna(value) or not str(value).strip():
                    continue
                multilingual_rows.append(
                    {
                        "id": f"local-multilingual-{row_index:05d}-{language}",
                        "source": "dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset",
                        "text": str(value),
                        "label": str(row["labels"]).lower().strip(),
                        "language": language,
                        "split": "unassigned",
                    }
                )
        multilingual_frame = pd.DataFrame(multilingual_rows)
        multilingual_scrubbed, counts = scrub_dataframe(multilingual_frame)
        multilingual_path = SCRUBBED_DIR / "h9_multilingual_selected_languages__scrubbed.csv"
        multilingual_scrubbed.to_csv(multilingual_path, index=False)
        combined_frames.append(multilingual_scrubbed)
        summary_rows.append(
            {
                "input_path": "in_memory_hf_multilingual_selected_languages",
                "output_path": str(multilingual_path),
                "rows": len(multilingual_frame),
                **counts,
            }
        )

summary = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / "h9_pii_scrub_summary.csv"
summary.to_csv(summary_path, index=False)

if combined_frames:
    combined = pd.concat(combined_frames, ignore_index=True)
    combined = combined.drop_duplicates(subset=["source", "text", "label", "language"])
    combined_path = PROCESSED_DIR / "h9_local_sms_corpus_scrubbed.csv"
    combined.to_csv(combined_path, index=False)
    print("Combined scrubbed corpus:", combined.shape, combined_path)
    display(combined.groupby(["source", "language", "label"]).size().reset_index(name="rows").head(30))
else:
    print("No normalized text/label corpora found for combined output.")

print("Scrub summary:", summary_path)
display(summary)
